# Comparison of LLM Fine-Tuning Approaches

This notebook compares different approaches to fine-tuning language models, focusing on full fine-tuning vs. parameter-efficient methods.

## Setup Environment

First, let's install the required packages:

In [ ]:
!pip install -q transformers peft datasets torch accelerate bitsandbytes

## Import Libraries

In [ ]:
import torch
import json
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

## Overview of Fine-Tuning Approaches

### 1. Full Fine-Tuning
- Updates all model parameters
- Requires significant computational resources
- Often provides the best performance
- High memory requirements

### 2. Parameter-Efficient Fine-Tuning (PEFT)
- Updates only a small subset of parameters
- Significantly reduces computational requirements
- Nearly matches full fine-tuning performance
- Popular methods include LoRA, QLoRA, Adapters, and Prefix Tuning

## Comparison of Memory Requirements

Let's compare the memory requirements of different approaches:

In [ ]:
def calculate_memory_requirements(model_size_params, approach):
    """Calculate approximate memory requirements for different approaches."""
    if approach == "full":
        # Full fine-tuning: model parameters (fp16) + optimizer states + gradients
        return (model_size_params * 2) + (model_size_params * 12) + (model_size_params * 2)
    elif approach == "lora":
        # LoRA: model parameters (fp16) + small trainable parameters + optimizer states for trainable params
        trainable_params = model_size_params * 0.01  # Approximately 1% of parameters
        return (model_size_params * 2) + (trainable_params * 2) + (trainable_params * 12)
    elif approach == "qlora":
        # QLoRA: model parameters (4-bit) + small trainable parameters + optimizer states for trainable params
        trainable_params = model_size_params * 0.01  # Approximately 1% of parameters
        return (model_size_params * 0.5) + (trainable_params * 2) + (trainable_params * 12)

# Model sizes in billions of parameters
model_sizes = [1, 7, 13, 33, 65]

print("Memory Requirements (GB):")
print("-" * 50)
print(f"{'Model Size (B)':15} {'Full Fine-tuning':20} {'LoRA':15} {'QLoRA':15}")
print("-" * 50)

for size in model_sizes:
    params = size * 1e9  # Convert billions to actual parameter count
    full_gb = calculate_memory_requirements(params, "full") / 1e9
    lora_gb = calculate_memory_requirements(params, "lora") / 1e9
    qlora_gb = calculate_memory_requirements(params, "qlora") / 1e9
    
    print(f"{size:15.1f} {full_gb:20.1f} {lora_gb:15.1f} {qlora_gb:15.1f}")

## Comparison of Training Time

Let's compare the relative training time of different approaches:

In [ ]:
def calculate_relative_training_time(model_size, approach):
    """Calculate relative training time for different approaches (normalized to full fine-tuning)."""
    if approach == "full":
        return 1.0  # Baseline
    elif approach == "lora":
        # LoRA is faster due to fewer parameter updates
        return 0.6
    elif approach == "qlora":
        # QLoRA is even faster due to quantization
        return 0.4

print("Relative Training Time (normalized to full fine-tuning):")
print("-" * 50)
print(f"{'Approach':15} {'Relative Time':20}")
print("-" * 50)

approaches = ["full", "lora", "qlora"]
for approach in approaches:
    relative_time = calculate_relative_training_time(7, approach)  # Using 7B model as reference
    print(f"{approach:15} {relative_time:20.2f}")

## Detailed Comparison of Approaches

### Full Fine-Tuning

**Pros:**
- Maximum performance potential
- No architectural constraints
- Well-established methodology

**Cons:**
- Extremely high memory requirements
- Requires high-end GPUs/TPUs
- Long training times
- Large storage requirements for checkpoints

**When to use:**
- When you have access to substantial computing resources
- For smaller models (< 1B parameters)
- When maximum performance is critical
- For production-grade models

### LoRA (Low-Rank Adaptation)

**Pros:**
- Significantly reduced memory requirements
- Faster training
- Performance close to full fine-tuning
- Small adapter size (easy to share and store)

**Cons:**
- Slightly lower performance than full fine-tuning
- Still requires 16-bit precision for base model
- Limited to certain model architectures

**When to use:**
- For medium to large models (1B-13B parameters)
- When working with consumer-grade GPUs (16-24GB VRAM)
- When training multiple specialized versions of a model
- For rapid prototyping

### QLoRA (Quantized Low-Rank Adaptation)

**Pros:**
- Dramatically reduced memory requirements
- Enables fine-tuning of very large models on consumer hardware
- Performance comparable to full fine-tuning
- Small adapter size

**Cons:**
- Slightly lower performance than full fine-tuning
- Quantization can introduce minor artifacts
- More complex implementation

**When to use:**
- For large models (7B-65B+ parameters)
- When working with limited GPU resources
- When fine-tuning on a single consumer GPU
- For efficient deployment of multiple specialized models

## Code Examples for Different Approaches

### 1. Full Fine-Tuning Example

In [ ]:
# This is a code snippet for full fine-tuning
'''
from transformers import Trainer, TrainingArguments

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Set up training arguments
training_args = TrainingArguments(
    output_dir="./full-fine-tuned-model",
    learning_rate=5e-5,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    save_steps=500,
    save_total_limit=2,
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
)

# Start training
trainer.train()
'''

### 2. LoRA Fine-Tuning Example

In [ ]:
# This is a code snippet for LoRA fine-tuning
'''
from peft import LoraConfig, get_peft_model, TaskType

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Define LoRA configuration
lora_config = LoraConfig(
    r=8,                    # Rank of the update matrices
    lora_alpha=16,          # Scaling factor
    lora_dropout=0.05,      # Dropout probability
    bias="none",            # Don't train bias parameters
    task_type=TaskType.CAUSAL_LM,
    target_modules=["c_attn"]  # Target the attention modules
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Set up training arguments
training_args = TrainingArguments(
    output_dir="./lora-fine-tuned-model",
    learning_rate=3e-4,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    save_steps=500,
    save_total_limit=2,
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
)

# Start training
trainer.train()
'''

### 3. QLoRA Fine-Tuning Example

In [ ]:
# This is a code snippet for QLoRA fine-tuning
'''
from transformers import BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model, TaskType

# Set up quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# Load the base model with quantization
model = AutoModelForCausalLM.from_pretrained(
    "EleutherAI/pythia-1.4b",
    quantization_config=bnb_config,
    device_map="auto"
)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-1.4b")

# Prepare the model for k-bit training
model = prepare_model_for_kbit_training(model)

# Define LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["query_key_value"]  # Target the attention modules
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Set up training arguments
training_args = TrainingArguments(
    output_dir="./qlora-fine-tuned-model",
    learning_rate=2e-4,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    save_steps=500,
    save_total_limit=2,
    gradient_checkpointing=True,
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
)

# Start training
trainer.train()
'''

## Performance Comparison

Let's compare the performance of different approaches on a thesaurus task:

In [ ]:
# This is a hypothetical performance comparison based on typical results
import matplotlib.pyplot as plt
import numpy as np

# Hypothetical performance metrics (higher is better)
approaches = ['Full Fine-tuning', 'LoRA', 'QLoRA']
performance = [0.95, 0.92, 0.90]  # Normalized performance scores
memory = [100, 25, 10]  # Relative memory usage
training_time = [100, 60, 40]  # Relative training time

# Create a bar chart for performance
plt.figure(figsize=(10, 6))
plt.bar(approaches, performance, color=['blue', 'green', 'orange'])
plt.ylim(0.8, 1.0)
plt.title('Performance Comparison')
plt.ylabel('Normalized Performance')
plt.grid(axis='y', linestyle='--', alpha=0.7)

for i, v in enumerate(performance):
    plt.text(i, v-0.02, f"{v:.2f}", ha='center', fontweight='bold')

plt.show()

# Create a bar chart for memory usage and training time
fig, ax1 = plt.subplots(figsize=(10, 6))

x = np.arange(len(approaches))
width = 0.35

ax1.bar(x - width/2, memory, width, label='Relative Memory Usage', color='red')
ax1.set_ylabel('Relative Memory Usage (%)')
ax1.set_title('Resource Usage Comparison')
ax1.set_xticks(x)
ax1.set_xticklabels(approaches)

ax2 = ax1.twinx()
ax2.bar(x + width/2, training_time, width, label='Relative Training Time', color='purple')
ax2.set_ylabel('Relative Training Time (%)')

# Add legend
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

fig.tight_layout()
plt.show()

## Choosing the Right Approach

Here's a decision tree to help you choose the right approach:

1. **Do you have access to high-end GPUs with >40GB VRAM?**
   - Yes → Consider full fine-tuning for maximum performance
   - No → Go to step 2

2. **What size is your model?**
   - Small (<1B parameters) → Full fine-tuning may be possible
   - Medium (1B-7B parameters) → Consider LoRA
   - Large (>7B parameters) → Use QLoRA

3. **How important is performance vs. resource efficiency?**
   - Performance is critical → Use full fine-tuning if possible, otherwise LoRA
   - Resource efficiency is critical → Use QLoRA

4. **Do you need to fine-tune multiple specialized versions?**
   - Yes → Use LoRA or QLoRA for efficient storage and swapping
   - No → Choose based on other criteria

## Conclusion

In this tutorial, you've learned about different approaches to fine-tuning language models:

1. **Full Fine-Tuning**: Updates all model parameters, provides maximum performance but requires significant resources.

2. **LoRA**: Updates only low-rank matrices, significantly reduces memory requirements while maintaining good performance.

3. **QLoRA**: Combines quantization with LoRA, dramatically reduces memory requirements, enabling fine-tuning of large models on consumer hardware.

Each approach has its strengths and weaknesses, and the best choice depends on your specific requirements, available resources, and the size of the model you're working with.

## References

- [LoRA Paper](https://arxiv.org/abs/2106.09685)
- [QLoRA Paper](https://arxiv.org/abs/2305.14314)
- [PEFT Library Documentation](https://huggingface.co/docs/peft/index)
- [Hugging Face Transformers](https://huggingface.co/docs/transformers/index)
- [Parameter-Efficient Fine-Tuning Methods](https://huggingface.co/blog/peft)